# MyoMap AI — DINOv2 Best (Self-Supervised + Efficient)
21M DINOv2-small (pretrained 142M images) + 2M fusion = 23M (92% smaller than 310M baseline).
Offline via `pretrained/dinov2-small` (no HF download, Internet OFF OK for scored submission).
Expected +0.03 AUROC vs EfficientNet-B0, +0.02 vs ConvNeXt.

In [ ]:
import pathlib, shutil
from pathlib import Path
found = list(Path("/kaggle/input").rglob("train.py"))
src_root = found[0].parents[1] if found else None
dst = Path("/kaggle/working/myomap-ai")
if dst.exists(): shutil.rmtree(dst)
shutil.copytree(src_root, dst)
print("copied", dst, "dinov2", (dst/"pretrained/dinov2-small/config.json").exists())
print("model_dinov2.py", (dst/"src/model_dinov2.py").exists())


In [ ]:
from pathlib import Path
csvs = list(Path("/kaggle/input").rglob("train.csv"))
print(csvs[:2])
if csvs:
    import pandas as pd
    df=pd.read_csv(csvs[0])
    print(df.shape, df.columns.tolist()[:12])
    print(df[[c for c in df.columns if c not in ["StudyInstanceUID","report_text"]][:12]].mean().sort_values())


In [ ]:
!pip install -q timm==1.0.9 transformers==4.32.0 albumentations==1.4.0 pydicom==2.4.4 pylibjpeg==1.4.0 accelerate==0.33.0 opencv-python-headless==4.10.0.84 2>&1 | tail -n 5
print("deps dinov2 done")


In [ ]:
!PYTHONPATH=/kaggle/working/myomap-ai/src:$PYTHONPATH python /kaggle/working/myomap-ai/src/train.py --config /kaggle/working/myomap-ai/configs/config_dinov2.yaml --fold 0 2>&1 | tee /kaggle/working/train_dinov2.log
print("dinov2 train done")


In [ ]:
!PYTHONPATH=/kaggle/working/myomap-ai/src:$PYTHONPATH python /kaggle/working/myomap-ai/src/export_onnx.py --ckpt /kaggle/working/myomap-ai/models/best_fold0.pth --config /kaggle/working/myomap-ai/configs/config_dinov2.yaml --out /kaggle/working/myomap-dinov2.onnx 2>&1 | tail -n 10
!ls -lh /kaggle/working/myomap-dinov2.onnx 2>&1 | head
!ls -lh /kaggle/working/myomap-ai/models/best_fold0.pth 2>&1 | head


In [ ]:
!PYTHONPATH=/kaggle/working/myomap-ai/src:$PYTHONPATH python /kaggle/working/myomap-ai/scripts/submission.py --ckpt /kaggle/working/myomap-ai/models/best_fold0.pth --config /kaggle/working/myomap-ai/configs/config_dinov2.yaml --out /kaggle/working/submission_dinov2.csv 2>&1 | tail -n 15
!head /kaggle/working/submission_dinov2.csv 2>&1 | head
!wc -l /kaggle/working/submission_dinov2.csv
